# Scraper Camptocamp.org — Voies d'escalade

Ce notebook récupère les voies d'escalade et leurs caractéristiques depuis
l'API publique de [camptocamp.org](https://www.camptocamp.org), pour
constituer un jeu de données exploitable en analyse.

## 1. Imports et configuration

In [ ]:
import csv
import os
import time
import requests
import pandas as pd
import math

# racine du endpoint de l'API
BASE_URL = "https://api.camptocamp.org"  

# constante python qui sera utilisée pour donner une valeur à un paramètre de l'API (langue des résultats)
LANG = "fr" 

# Nombre de voies récupérées par page lors du listing
PAGE_SIZE = 100

# Pause entre chaque appel réseau (en secondes), pour ne pas surcharger l'API
SLEEP_BETWEEN_CALLS = 0.3

# Colonnes qui vont être exportées dans le fichier CSV final
FIELDS = [
    "document_id", "title", "title_prefix", "quality", "activities",
    "climbing_outdoor_type", "global_rating", "engagement_rating",
    "risk_rating", "equipment_rating", "exposition_rock_rating",
    "rock_free_rating", "rock_required_rating", "aid_rating",
    "rock_types", "orientations", "configuration",
    "elevation_min", "elevation_max", "height_diff_up", "height_diff_down",
    "height_diff_difficulties", "durations", "description",
]

print("Configuration chargée.")


Configuration chargée.


## 2. Fonctions de récupération et de parsing

### 2.1 Fonctions auxiliaires

In [ ]:
# Récupère une page de la liste des voies d'escalade
def fetch_routes_page(offset, limit=PAGE_SIZE):
    params = {"act": "rock_climbing", "limit": limit, "offset": offset, "pl": LANG} # définition des paramètres de l'API pour la requête
    resp = requests.get(f"{BASE_URL}/routes", params=params, timeout=30) # envoie une requête GET à l'API pour récupérer la liste des voies d'escalade avec les paramètres définis
                                                                         # concrètement, la fonction transforme la f-string en une URL complète, puis envoie la requête GET à cette URL avec les paramètres définis. 
                                                                         # Le paramètre timeout=30 signifie que la requête sera annulée si elle prend plus de 30 secondes.
    resp.raise_for_status() # Check de la nature de la réponse de l'API. Cette réponse contient un code de statut HTTP. Si le code de statut indique une erreur (par exemple, 4xx ou 5xx), cette méthode lève une exception HTTPError. Cela permet de gérer les erreurs de manière appropriée dans le code appelant.
    return resp.json() # S'il n'y a pas d'erreur, retourne le contenu de la réponse sous forme de dictionnaire Python (la réponse est au format JSON, et la méthode json() la convertit en un dictionnaire Python).


def join_or_none(values):
    """Joint une liste avec ';', ou renvoie None si absente/vide."""
    if not values:
        return None
    return ";".join(values)


# Récupère le document DÉTAILLÉ d'une voie (/routes/{id}). Cela permet d'obtenir les paramètres "description", "configuration" et "rock_types".
def fetch_route_detail(document_id):

    url = f"{BASE_URL}/routes/{document_id}" # endpoint de l'API pour récupérer le détail d'une voie d'escalade spécifique, identifié par son document_id.
    resp = requests.get(url, params={"pl": LANG}, timeout=30)
    resp.raise_for_status()
    data = resp.json() # on sauve le contenu de la réponse sous forme de dictionnaire Python.

    description = None # initialisation de la variable description à None. Cela sert de valeur par défaut au cas où aucune description n'est trouvée dans les locales.
    locales = data.get("locales", []) # locales est une liste de dictionnaires, chacun représentant une version localisée (traduite) des informations de la voie. 
                                      # Ici, on récupère la liste des locales à partir du dictionnaire data. Si la clé "locales" n'existe pas, on obtient une liste vide par défaut.
    if locales:
        for loc in locales:
            if loc.get("lang") == LANG: # on cherche la locale correspondant à la langue définie par la constante LANG (ici, "fr" pour français).
                description = loc.get("description") # on récupère la description de la voie dans cette locale spécifique.
                break
        else:
            description = locales[0].get("description") # Si aucune locale ne correspond à la langue souhaitée, on prend la description de la première locale disponible.

    return {
        "description": description,
        "rock_types": join_or_none(data.get("rock_types")),
        "configuration": join_or_none(data.get("configuration")),
    }


# Mise en forme d'une ligne de données à partir d'un document JSON de l'API (une voie). On formate correctement chaque ligne du futur CSV. 
def parse_route_listing(doc):
    """Extrait les champs structurés d'un document de listing (/routes)."""
    row = {field: None for field in FIELDS} # initialisation d'un dictionnaire row avec toutes les clés définies dans FIELDS, et des valeurs initiales à None. 
                                            # Cela garantit que toutes les colonnes du CSV final seront présentes, même si certaines informations sont manquantes dans le document de listing.
    # On récupère d'abord toutes les données universelles (chiffres ou textes qui sont des codes internes : quality, climbing_outdoor_type...)
    row["document_id"] = doc.get("document_id")
    row["quality"] = doc.get("quality")
    row["activities"] = ";".join(doc.get("activities") or [])
    row["climbing_outdoor_type"] = doc.get("climbing_outdoor_type")
    row["global_rating"] = doc.get("global_rating")
    row["engagement_rating"] = doc.get("engagement_rating")
    row["risk_rating"] = doc.get("risk_rating")
    row["equipment_rating"] = doc.get("equipment_rating")
    row["exposition_rock_rating"] = doc.get("exposition_rock_rating")
    row["rock_free_rating"] = doc.get("rock_free_rating")
    row["rock_required_rating"] = doc.get("rock_required_rating")
    row["aid_rating"] = doc.get("aid_rating")
    row["orientations"] = ";".join(doc.get("orientations") or [])
    row["elevation_min"] = doc.get("elevation_min")
    row["elevation_max"] = doc.get("elevation_max")
    row["height_diff_up"] = doc.get("height_diff_up")
    row["height_diff_down"] = doc.get("height_diff_down")
    row["height_diff_difficulties"] = doc.get("height_diff_difficulties")
    row["durations"] = ";".join(doc.get("durations") or [])

    # On récupère ensuite les données traduites (localisées)
    locales = doc.get("locales") or []
    if locales:
        row["title"] = locales[0].get("title")
        row["title_prefix"] = locales[0].get("title_prefix")
    return row

Fonctions définies (version corrigée : rock_types/configuration lus depuis le détail).


### 2.2 Fonction principale du scraping

In [ ]:

# Boucle principale du scraping : pagine sur le listing, complète avec le détail et écrit le résultat au fur et à mesure dans un CSV.
def scrape(max_routes, output_path, fetch_descriptions=True, verbose=True):

    offset = 0
    total_written = 0
    total_available = None
    all_rows = []

    # On ouvre le fichier CSV en mode écriture, avec l'encodage UTF-8 et sans ligne vide supplémentaire entre les lignes.
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        writer.writeheader()

        while total_written < max_routes:
            page_limit = min(PAGE_SIZE, max_routes - total_written) # on calcule le nombre de voies à récupérer pour cette page, en prenant le minimum entre PAGE_SIZE et le nombre de voies restantes à récupérer (max_routes - total_written).
            if verbose: # paramètre très courant pour afficher ou non des messages de progression dans la console. Si verbose est True, on affiche le message suivant.
                print(f"[Listing] offset={offset}, limit={page_limit}...")

            data = fetch_routes_page(offset=offset, limit=page_limit) # on récupère une page de voies d'escalade à partir de l'API en appelant la fonction fetch_routes_page avec les paramètres offset et limit calculés précédemment.
            documents = data.get("documents", []) # on récupère la liste des documents (voies d'escalade) à partir du dictionnaire data renvoyé par l'API. Si la clé "documents" n'existe pas, on obtient une liste vide par défaut.
            total_available = data.get("total", total_available) # on récupère le nombre total de voies d'escalade disponibles à partir du dictionnaire data renvoyé par l'API. Si la clé "total" n'existe pas, on conserve la valeur précédente de total_available.

            if not documents:
                print("Plus de voies à récupérer.")
                break

            for doc in documents:
                row = parse_route_listing(doc) # on transforme chaque document de voie d'escalade en une ligne de données formatée pour le CSV en appelant la fonction parse_route_listing.
                if fetch_descriptions: # si le paramètre fetch_descriptions est True, on récupère les détails de la voie d'escalade (description, types de roches, configuration) en appelant la fonction fetch_route_detail avec le document_id de la voie.
                    try:
                        detail = fetch_route_detail(row["document_id"])
                        row["description"] = detail["description"]
                        row["rock_types"] = detail["rock_types"]
                        row["configuration"] = detail["configuration"]
                    except requests.RequestException as e:
                        print(f"  ! Erreur détail pour {row['document_id']}: {e}")
                    time.sleep(SLEEP_BETWEEN_CALLS)

                writer.writerow(row) # on écrit la ligne de données formatée dans le fichier CSV en utilisant l'objet writer créé précédemment. Chaque ligne correspond à une voie d'escalade.
                all_rows.append(row) # on ajoute la ligne de données formatée à la liste all_rows, qui contient toutes les lignes de données récupérées et formatées jusqu'à présent. Cela permet de conserver une copie en mémoire des données pour un traitement ultérieur si nécessaire.
                total_written += 1
                if total_written >= max_routes:
                    break

            if verbose:
                print(f"  -> {total_written} voies écrites (total disponible: {total_available})")
            offset += page_limit
            time.sleep(SLEEP_BETWEEN_CALLS)

            if total_available is not None and offset >= total_available:
                break

    print(f"\nTerminé. {total_written} voies écrites dans {output_path}")
    return all_rows

## 3. Test rapide (10 voies)

In [ ]:
rows_test = scrape(max_routes=10, output_path="test_10.csv", fetch_descriptions=True)

df_test = pd.DataFrame(rows_test)
df_test

## 4. Test intermédiaire (500 voies)

In [ ]:
rows_500 = scrape(max_routes=500, output_path="test_500.csv", fetch_descriptions=True)

df_500 = pd.DataFrame(rows_500)
df_500.head(20)


In [ ]:
# Quelques vérifications rapides sur l'échantillon de 500 voies

print("Nombre de voies récupérées :", len(df_500))
print("\nValeurs manquantes par colonne :")
print(df_500.isna().sum())
print("\nExemple de description :")
print(df_500['description'].dropna().iloc[0])


## 5. Récupération sur un échantillon (< 10 000 voies)

⚠️ Cette cellule utilise `scrape()`, qui pagine simplement par `offset`.
**Elle ne peut PAS dépasser 10 000 voies** (limite de l'API, voir section 7
pour la comprendre et la contourner). Utilise-la uniquement pour un
échantillon volontairement limité (ex. 5000 voies) ; pour tout récupérer,
va directement à la **section 7**.

In [ ]:
max_routes = 5000  # reste VOLONTAIREMENT sous 10000 (limite de l'API, voir section 7)

rows_sample = scrape(max_routes=max_routes, output_path="camptocamp_routes_sample.csv", fetch_descriptions=True)

df_sample = pd.DataFrame(rows_sample)
print("Nombre total de voies récupérées :", len(df_sample))
df_sample.head(20)


## 6. Récupération complète — contournement de la limite des 10 000 résultats

**Important** : l'API refuse toute requête avec `offset >= 10000`
(erreur `400 Bad Request`). C'est une limite technique du moteur de recherche (Elasticsearch) utilisé par Camptocamp, pas un bug de notre code.

**Solution** : on découpe automatiquement le monde en zones géographiques via le paramètre `bbox`), en redécoupant récursivement toute zone qui
contiendrait encore plus de 9000 voies, jusqu'à ce que chaque zone soit sous la limite. On peut alors paginer normalement dans chaque zone.

Cette section reprend aussi automatiquement là où elle s'est arrêtée en cas de plantage : elle relit les voies déjà présentes dans le fichier CSV et
les ignore.

### 6.1 Fonctions auxiliaires

In [ ]:
# L'API Camptocamp a un bug côté serveur : son filtre bbox est silencieusement ignoré si l'une des 4 coordonnées vaut exactement 0 (en Python, `all([...])`
# renvoie False si une valeur est 0). Comme notre bbox de départ part de longitude -180 à +180, la toute première coupe tombe pile sur x=0 (le
# méridien de Greenwich), ce qui casse le filtre pour toute une partie de l'arbre de découpage. On corrige ça en décalant très légèrement (1mm, en
# mètres) toute coordonnée qui vaudrait exactement 0 — sans impact réel sur la précision géographique.
BBOX_EPSILON = 1e-3

# On évite les coordonnées nulles pour contourner le bug de l'API (filtre bbox est silencieusement ignoré si l'une des 4 coordonnées vaut exactement 0)
def avoid_zero(v):
    return BBOX_EPSILON if v == 0 else v


# Convertit des coordonnées lon/lat (degrés) en Web Mercator (EPSG:3857, mètres) pour que le paramètre bbox de l'API Camptocamp fonctionne correctement.
def lonlat_to_mercator(lon, lat):
    x = lon * 20037508.34 / 180.0
    y = math.log(math.tan((90 + lat) * math.pi / 360.0)) / (math.pi / 180.0)
    y = y * 20037508.34 / 180.0
    return avoid_zero(x), avoid_zero(y)

# Appelle la fonction précédente pour convertir plusieurs coordonnées à la fois : le coin inférieur-gauche et le coin supérieur-droit qui définissent une bbox.
def lonlat_bbox_to_mercator(lon_min, lat_min, lon_max, lat_max):
    x_min, y_min = lonlat_to_mercator(lon_min, lat_min)
    x_max, y_max = lonlat_to_mercator(lon_max, lat_max)
    return (x_min, y_min, x_max, y_max)

# Récupère une page de la liste des voies d'escalade, dnas une certaine zone géographique (avec un paramètre bbox optionnel mais utilisé dans notre cas).
def fetch_routes_page(offset, limit=PAGE_SIZE, bbox=None):
    params = {"act": "rock_climbing", "limit": limit, "offset": offset, "pl": LANG}
    if bbox is not None:
        params["bbox"] = ",".join(str(v) for v in bbox)
    resp = requests.get(f"{BASE_URL}/routes", params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

# Renvoie le nombre total de voies dans une zone donnée (1 seul appel léger).
def get_bbox_count(bbox):
    data = fetch_routes_page(offset=0, limit=1, bbox=bbox)
    return data.get("total", 0)

# Découpe récursivement une bbox en 4 quadrants tant que le nombre de voies qu'elle contient dépasse max_count.
def find_leaf_bboxes(bbox, max_count=9000, depth=0, max_depth=12, verbose=True):
    total = get_bbox_count(bbox)
    if total == 0:
        return []
    if total <= max_count or depth >= max_depth:
        if verbose:
            print(f"  zone retenue : {total} voies (profondeur {depth})")
        return [(bbox, total)]

    x_min, y_min, x_max, y_max = bbox
    x_mid, y_mid = avoid_zero((x_min + x_max) / 2), avoid_zero((y_min + y_max) / 2)
    quadrants = [
        (x_min, y_min, x_mid, y_mid),
        (x_mid, y_min, x_max, y_mid),
        (x_min, y_mid, x_mid, y_max),
        (x_mid, y_mid, x_max, y_max),
    ]
    leaves = []
    for q in quadrants:
        leaves.extend(find_leaf_bboxes(q, max_count, depth + 1, max_depth, verbose))
    return leaves

Fonctions de récupération par découpage géographique définies.


### 6.2 Fonction principale du scraping

In [ ]:
# Scraping complet via découpage géographique, avec reprise automatique en cas de relance.
def scrape_via_bbox(output_path, fetch_descriptions=True, max_count_per_leaf=9000, verbose=True):
    # 1. Charger les voies déjà récupérées (pour reprise / dédoublonnage)
    processed_ids = set()
    file_exists = os.path.exists(output_path)
    if file_exists:
        with open(output_path, encoding="utf-8") as f:
            for row in csv.DictReader(f):
                processed_ids.add(row["document_id"])
        print(f"{len(processed_ids)} voies déjà présentes dans {output_path} (seront ignorées).")

    # 2. Découper le monde en zones sous la limite
    print("Découpage géographique en cours...")
    world_bbox = lonlat_bbox_to_mercator(-180, -60, 180, 85)
    leaves = find_leaf_bboxes(world_bbox, max_count=max_count_per_leaf, verbose=verbose)
    print(f"\n{len(leaves)} zones à parcourir, {sum(t for _, t in leaves)} voies au total (doublons de bord possibles, dédoublonnés automatiquement).\n")

    mode = "a" if file_exists else "w"
    total_written = len(processed_ids)

    with open(output_path, mode, newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        if not file_exists:
            writer.writeheader()

        for zone_num, (leaf_bbox, leaf_total) in enumerate(leaves, start=1):
            offset = 0
            while offset < leaf_total:
                limit = min(PAGE_SIZE, leaf_total - offset)
                try:
                    data = fetch_routes_page(offset=offset, limit=limit, bbox=leaf_bbox)
                except requests.RequestException as e:
                    print(f"  ! Erreur zone {zone_num}, offset {offset}: {e}")
                    break

                documents = data.get("documents", [])
                if not documents:
                    break

                for doc in documents:
                    doc_id = str(doc.get("document_id"))
                    if doc_id in processed_ids:
                        continue  # déjà récupéré lors d'une exécution précédente ou zone adjacente

                    row = parse_route_listing(doc)
                    if fetch_descriptions:
                        try:
                            detail = fetch_route_detail(row["document_id"])
                            row["description"] = detail["description"]
                            row["rock_types"] = detail["rock_types"]
                            row["configuration"] = detail["configuration"]
                        except requests.RequestException as e:
                            print(f"  ! Erreur détail {row['document_id']}: {e}")
                        time.sleep(SLEEP_BETWEEN_CALLS)

                    writer.writerow(row)
                    f.flush()
                    processed_ids.add(doc_id)
                    total_written += 1

                offset += limit
                time.sleep(SLEEP_BETWEEN_CALLS)

            if verbose:
                print(f"[Zone {zone_num}/{len(leaves)}] terminée ({leaf_total} voies dans la zone). Total cumulé écrit : {total_written}")

    print(f"\nTerminé. {total_written} voies uniques écrites dans {output_path}")
    return total_written

In [ ]:
scrape_via_bbox(output_path="camptocamp_routes_full.csv", fetch_descriptions=True)

Découpage géographique en cours...
  zone retenue : 1605 voies (profondeur 1)
  zone retenue : 70 voies (profondeur 2)
  zone retenue : 15 voies (profondeur 2)
  zone retenue : 13 voies (profondeur 3)
  zone retenue : 15 voies (profondeur 3)
  zone retenue : 47 voies (profondeur 4)
  zone retenue : 111 voies (profondeur 4)
  zone retenue : 159 voies (profondeur 5)
  zone retenue : 140 voies (profondeur 5)
  zone retenue : 1739 voies (profondeur 6)
  zone retenue : 677 voies (profondeur 6)
  zone retenue : 3141 voies (profondeur 6)
  zone retenue : 5774 voies (profondeur 6)
  zone retenue : 132 voies (profondeur 5)
  zone retenue : 54 voies (profondeur 4)
  zone retenue : 66 voies (profondeur 3)
  zone retenue : 36 voies (profondeur 2)
  zone retenue : 283 voies (profondeur 1)
  zone retenue : 2822 voies (profondeur 1)

19 zones à parcourir, 16899 voies au total (doublons de bord possibles, dédoublonnés automatiquement).

[Zone 1/19] terminée (1605 voies dans la zone). Total cumulé écri

16860

## 7. Chargement des données

In [9]:
df = pd.read_csv("camptocamp_routes_full.csv")
df

,document_id,title,title_prefix,quality,activities,climbing_outdoor_type,global_rating,engagement_rating,risk_rating,equipment_rating,...,rock_types,orientations,configuration,elevation_min,elevation_max,height_diff_up,height_diff_down,height_diff_difficulties,durations,description
0,1948187,Triangle des Coutchets - Brouillard d'été,Pic des Coutchets,draft,rock_climbing,multi,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A gauche de Aquim
1,1948176,Triangle des Coutchets - Soleil d'automne,Pic des Coutchets,medium,rock_climbing,multi,TD-,NaN,NaN,P2,...,calcaire,NE,face,NaN,NaN,NaN,NaN,150.0,1,Approche: monter au plateau de Bezou soit à pi...
2,1944609,Capricho,Aiguille de la Canalona,medium,rock_climbing,multi,D+,NaN,NaN,P2+,...,calcaire,S,pillar,1850.0,2540.0,690.0,NaN,170.0,1,"## Approche ##\n\nDepuis le Câble, on suit la ..."
3,1944282,Secteur Rocaïne - Cash cache,Les Douanes,fine,rock_climbing,single,NaN,NaN,NaN,NaN,...,calcaire,S,NaN,1560.0,1610.0,100.0,100.0,15.0,1,## Approche # 15min\nDu parking en bordure de ...
4,1944020,Face W - Heureux qui comme Ulysse,Pène Sarrière,fine,rock_climbing,multi,ED-,I,X2,P1,...,calcaire,W,face,1370.0,1830.0,NaN,NaN,195.0,1,## Approche # 45min\nSuivre l'approche de la [...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16855,54116,L'avenue des chamois,Motteux Péteux,fine,rock_climbing,multi,AD+,II,NaN,P1+,...,calcaire,S,face,1633.0,2232.0,680.0,680.0,300.0,1,"## Approche # 1h15\nDe Mittelberg (1632 m), su..."
16856,53904,Voie Originale de l'Y,Miroir d'Argentine,medium,rock_climbing,multi,D-,II,NaN,P1+,...,NaN,N,face,1469.0,2325.0,850.0,892.0,380.0,1,"## Approche # (env. 1h)\nDu parking supérieur,..."
16857,53903,Papageno,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,NaN,NW,face,1462.0,2325.0,800.0,887.0,400.0,1,"## Approche\nDu parking, traverser le torrent ..."
16858,53893,Voie Directe,Miroir d'Argentine,fine,rock_climbing,multi,D,II,NaN,P2,...,calcaire,NW,face,1462.0,2325.0,800.0,800.0,400.0,1,[img=1045456 right]Tracé de la voie[/img]\n\n#...
